# Prompt Engineering & In-Context Learning

Companion notebook for the [Prompt Engineering lesson](https://ml-viz-ruby.vercel.app/courses/building-with-llms/01-prompt-engineering).

**The idea in one sentence.** An LLM just samples the next token from a probability
distribution, so the two levers you control at inference — the **prompt** (which shifts
the distribution) and the **decoding parameters** (temperature, top-p) — are what shape
its behaviour without touching a single weight.

The decoding knobs:

- **Temperature** rescales the logits: low $T$ sharpens (near-deterministic), high $T$
  flattens (more random).
- **Top-p (nucleus)** keeps the smallest set of tokens whose cumulative probability
  reaches $p$, then renormalises — adaptive truncation.

We implement both from scratch, **validate the softmax and nucleus rules**, then cover
the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## Decoding from a next-token distribution

An LLM emits **logits** over the vocabulary; decoding turns them into text. We implement temperature scaling and top-p (nucleus) sampling on a fixed toy distribution.

In [ ]:
TOKENS = ['mat','floor','sofa','roof','table','bed','grass','moon']
LOGITS = np.array([3.2, 2.6, 2.1, 1.0, 1.7, 1.3, 0.4, -0.5])

def softmax_t(logits, T):
    z = logits / max(T, 1e-3)
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

for T in [0.5, 1.0, 1.5]:
    p = softmax_t(LOGITS, T)
    print(f'T={T}: ' + '  '.join(f'{t}={pi:.2f}' for t, pi in zip(TOKENS, p)))

### Validate: temperature sharpens or flattens the distribution

Lower temperature concentrates probability mass (lower entropy → more deterministic);
higher temperature spreads it out (higher entropy → more diverse). We confirm entropy
increases monotonically with temperature and that our softmax is a valid distribution.

In [ ]:
def entropy(p): return float(-(p * np.log(p + 1e-12)).sum())
for T in [0.5, 1.0, 1.5]:
    p = softmax_t(LOGITS, T)
    assert abs(p.sum() - 1.0) < 1e-9, 'softmax must sum to 1'
    print(f'T={T}: entropy = {entropy(p):.3f} bits, top-token prob = {p.max():.3f}')
assert entropy(softmax_t(LOGITS, 0.5)) < entropy(softmax_t(LOGITS, 1.5)), 'higher T -> higher entropy'
print('\n✅ temperature is the sharpness dial: low = deterministic, high = diverse')

Low temperature concentrates mass on the top token (greedy); high temperature flattens it. Let's visualise.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for ax, T in zip(axes, [0.5, 1.0, 1.5]):
    p = softmax_t(LOGITS, T)
    ax.bar(TOKENS, p, color='#6366f1')
    ax.set_title(f'temperature = {T}')
    ax.tick_params(axis='x', rotation=45)
axes[0].set_ylabel('probability')
plt.tight_layout(); plt.show()

## Top-p (nucleus) sampling

Keep the smallest set of tokens whose cumulative probability reaches `p`, drop the tail, renormalise.

In [ ]:
def nucleus(probs, p):
    order = np.argsort(probs)[::-1]
    cum = np.cumsum(probs[order])
    cutoff = np.searchsorted(cum, p) + 1  # how many to keep
    keep = order[:cutoff]
    out = np.zeros_like(probs)
    out[keep] = probs[keep] / probs[keep].sum()
    return out

p = softmax_t(LOGITS, 1.0)
q = nucleus(p, 0.9)
print('kept tokens:', [TOKENS[i] for i in np.where(q > 0)[0]])
print('renormalised:', np.round(q[q > 0], 3))

### Validate: nucleus sampling keeps the smallest mass-$p$ set and renormalises

Top-p keeps the fewest top tokens whose cumulative probability first reaches $p$, zeros
the rest, and renormalises so the survivors sum to 1. We confirm the kept set's original
mass was $\ge p$ and that the output is a valid distribution — and that it's *adaptive*
(a peakier input keeps fewer tokens).

In [ ]:
p_full = softmax_t(LOGITS, 1.0)
q = nucleus(p_full, 0.9)
kept = np.where(q > 0)[0]
print(f'kept {len(kept)} tokens, original mass = {p_full[kept].sum():.3f}, renormalised sum = {q.sum():.3f}')
assert p_full[kept].sum() >= 0.9 - 1e-9, 'the kept set must cover at least p of the mass'
assert abs(q.sum() - 1.0) < 1e-9, 'nucleus output must renormalise to 1'
# adaptive: a peakier distribution (low T) keeps fewer tokens at the same p
kept_sharp = np.count_nonzero(nucleus(softmax_t(LOGITS, 0.3), 0.9))
kept_flat  = np.count_nonzero(nucleus(softmax_t(LOGITS, 1.5), 0.9))
print(f'tokens kept at p=0.9 — sharp dist: {kept_sharp}, flat dist: {kept_flat}')
assert kept_sharp <= kept_flat, 'top-p adapts: a peaky distribution needs fewer tokens'
print('\n✅ nucleus keeps the adaptive smallest set covering mass p, renormalised')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **high temperature, no truncation** | samples implausible tokens → incoherent output; pair with top-p |
| **temperature 0 isn't always best** | greedy can loop/repeat on open-ended text |
| **prompt sensitivity** | small wording changes shift the distribution a lot; test prompts empirically |
| **few-shot ordering** | example order and recency bias the model |
| **context length** | in-context examples cost tokens and can crowd out the real query |

Demo: a few-shot prompt reshapes the next-token distribution without any weight update.

In [ ]:
# The prompt shifts the distribution BEFORE decoding — that's in-context learning. We
# simulate it: a few-shot 'prompt' that up-weights the on-task tokens changes which token
# decoding selects, without any weight update.
prior = softmax_t(LOGITS, 1.0)
# a few-shot prompt biases toward 'table'/'floor' (furniture context) via a logit bump
context_bump = np.array([0, 1.5, 1.0, 0, 2.0, 1.0, 0, 0])   # in-context evidence
posterior = softmax_t(LOGITS + context_bump, 1.0)
print('argmax without context:', TOKENS[prior.argmax()])
print('argmax with few-shot context:', TOKENS[posterior.argmax()])
assert prior.argmax() != posterior.argmax() or posterior.max() > prior.max()
print('\nThe prompt reshapes the next-token distribution -> in-context learning is Bayesian conditioning.')

## ✏️ Your turn

Implement **top-k** sampling: keep only the `k` highest-probability tokens, then renormalise.

In [ ]:
def top_k(probs, k):
    # TODO(you): zero out all but the k largest probabilities, then renormalise.
    out = np.zeros_like(probs)
    # ...
    return out

res = top_k(softmax_t(LOGITS, 1.0), 3)
assert np.count_nonzero(res) == 3
assert abs(res.sum() - 1.0) < 1e-9
print('passed ✓')

<details><summary>Solution</summary>

```python
def top_k(probs, k):
    idx = np.argsort(probs)[::-1][:k]
    out = np.zeros_like(probs)
    out[idx] = probs[idx] / probs[idx].sum()
    return out
```

</details>

## Key takeaways

- **An LLM samples the next token from a distribution;** the prompt and decoding
  parameters are your two weight-free control levers.
- **Temperature is the sharpness dial** — low = deterministic, high = diverse (verified
  via entropy).
- **Top-p keeps the adaptive smallest mass-$p$ set** and renormalises (verified) — the
  open-ended-generation default.
- **The prompt shifts the distribution before decoding** — that's in-context learning,
  a form of Bayesian conditioning (demo).